In [14]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
from attention import SmartDataCollator, TextClassificationDataset, EncoderModel, SkipConnection, evaluate_model, train_model, print_model_parameters

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [16]:
ds = load_dataset("stanfordnlp/sst2")

In [17]:
tokenizer = Tokenizer.from_file(
    f"results/bpe_{8000}.json"
)

Change the number to change which tokenizer is used.

In [18]:
embedding_dim = 512
hidden_dim = 128
output_dim = 512
batch_size = 128

In [19]:
collate_fn = SmartDataCollator(tokenizer)

dataset_train = TextClassificationDataset(list(ds["train"]["sentence"]), list(ds["train"]["label"]), tokenizer)
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

dataset_val = TextClassificationDataset(list(ds["validation"]["sentence"]), list(ds["validation"]["label"]), tokenizer)
dataloader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

For the test dataset, do the same as above

In [ ]:
MLPmodel = nn.Sequential(
    EncoderModel(
        vocab_size=tokenizer.get_vocab_size(),
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        pad_idx=tokenizer.token_to_id("[PAD]")
    ),
    SkipConnection(output_dim, output_dim // 2, nn.GELU()),
    nn.GELU(),
    SkipConnection(output_dim // 2, 1, nn.GELU()),
    nn.Sigmoid()
).to(device)
MLPoptimizer = torch.optim.AdamW(MLPmodel.parameters(), lr=1e-4)

In [21]:
print_model_parameters(MLPmodel)

Totale parameters: 13,278,722
Trainbare parameters: 13,278,722


In [ ]:
Linearmodel = nn.Sequential(
    EncoderModel(
        vocab_size=tokenizer.get_vocab_size(),
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        pad_idx=tokenizer.token_to_id("[PAD]")
    ),
    nn.Linear(output_dim, output_dim // 2),
    nn.GELU(),
    nn.Linear(output_dim // 2, 1),
    nn.Sigmoid()
).to(device)
Linearoptimizer = torch.optim.AdamW(Linearmodel.parameters(), lr=1e-4)

In [23]:
print_model_parameters(Linearmodel)

Totale parameters: 12,818,689
Trainbare parameters: 12,818,689


For different depths you can add more layers.  
You can change the parameters and learning rate.

In [ ]:
criterion = nn.BCEWithLogitsLoss()

AttributeError: module 'torch.nn' has no attribute 'NLLoss'

In [ ]:
MLPmodel = train_model(MLPmodel, MLPoptimizer, dataloader_train, num_epochs=10, criterion=criterion, val_loader=dataloader_val, device=device)

Epoch 1/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 2/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 3/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 4/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 5/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 6/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 7/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 8/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 9/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 10/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%


In [ ]:
Linearmodel = train_model(Linearmodel, Linearoptimizer, dataloader_train, num_epochs=10, criterion=criterion, val_loader=dataloader_val, device=device)

Epoch 1/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 2/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 3/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 4/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 5/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 6/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 7/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 8/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%
Epoch 9/10 | Training Loss: 0.7554 | Validation Loss: 0.8041 | Validation Accuracy: 50.92%


KeyboardInterrupt: 